# CIT.M1 - Molecular Data Acquisition and Representation
### Notebook 02. PUG-REST and HTTP-POST Requests 

**Version 1.0.0 - February, 2026. Monterrey**

**Author:** [Flavio F. Contreras-Torres](https://orcid.org/my-orcid?orcid=0000-0003-2375-131X). Tecnológico de Monterrey.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/CheminformaticsTutorial/blob/main/01_Data_Acquisition/notebooks/02_pugrest_post_requests.ipynb)


---


### Contents

This notebook extends the molecular data acquisition workflow by introducing **HTTP POST requests** in **PubChem PUG-REST**, a method required when simple text-based searches are not sufficient. Instead of querying PubChem only by name or CID, you will learn how to search using the actual **chemical structure** of a molecule.

The activities are structured to guide you through the process of:

1. Structure-Based Querying: Learn how to use chemical representations such as SMILES stringd or SDF files as the starting point for molecular searches.

2. GET vs. POST Logic: Understand the difference between sending parameters in the **URL** versus sending data in the **request body**, and why POST is essential for more complex structural queries.

3. Mastering `requests.post()`: Implement Python workflows to submit structural queries, upload molecular files, and retrieve structured responses from PubChem.

4. Flexible Data Retrieval: Request multiple molecular properties in a single call and organize the results into research-ready tables.

5. Advanced Query Handling: Introduce strategies for processing larger inputs, asynchronous queries, and more robust API interactions.


This notebook is designed to be completed in approximately **120–180 minutes**.  

---


### How to work

1. **Open the notebook**: Click on the "Open in Colab" button or use the link provided to open the tutorial in **Google Colab**.
2. **Create your workspace**: Once open, go to **File > Save a copy in Drive**. This is vital! You must work on your own copy to ensure your progress is saved.
    - **Tip**: Look at the top-left corner. If you see "Copy of...", you are ready to go!
3. **Solve the exercises**: Complete the missing parts of the code where indicated. You can run each cell to test your progress (Shift + Enter).
4. When you finish:
    - **Rename** your file following the convention:
        - `YourID_NB2_M1.ipynb`
    - **Download the file**: Go to File > Download > Download .ipynb.
    - **Upload** the downloaded file to the **CANVAS assignment module**.

**Do not** modify the notebook structure or function signatures unless explicitly stated.


---

## The HTTP POST Method

**PUG-REST** (Power User Gateway-Representational State Transfer) handles short requests where results are provided in a single call with a maximum duration of 30 seconds. This eliminates the need for intermediate steps to check if the request is complete. PUG-REST requests can be executed using either **HTTP-GET** or **HTTP-POST** methods. These methods offer different levels of cybersecurity and varying degrees of convenience, depending on the specific requirements of the research task.

In our previous notebook, we asked `PubChem` for data using a simple URL `(HTTP GET)`. This is like calling a library and asking for a book by its title (i.e. "the header"). 

But what if you have a **map** of a new molecule you designed? Or a complex file containing rare characters (**InChIKey**) or 2D or 3D atomic coordinates (**SDF**)? These "packages" are too big to fit in a URL. To handle this, we use the `HTTP POST` method. 

**Think of it this way:**

* **HTTP GET:** Sending a postcard (everyone sees it, it must be short).
* **HTTP POST:** Sending a sealed parcel (it can carry heavy files and complex instructions).
 


## GET vs. POST: Choosing the Right Tool for the Task

The key differences between these two methods are shown in Table 1. Understanding these distinctions is crucial for deciding how to structure your requests to PubChem depending on the complexity and sensitivity of your research data.

### Table 1. Differences between HTTP GET and POST methods

| Feature | HTTP GET | HTTP POST |
| :--- | :--- | :--- |
| **Data Transmission** | Sends variables encoded directly in the URL to execute the request. | Sends variables as part of the data "payload"; therefore, they are not visible in the URL. |
| **Security** | **Less secure:** Data is visible in the URL and is stored in browser history and web server logs. | **More secure:** Data is not stored in browser history or server logs, making it harder to intercept. |
| **Size Limit** | Average limit of **2048 characters** for the entire URL. | **No limit** for the size of the data being sent. |
| **Character Types** | Supports only **ASCII characters** (0-9, English alphabet, and basic symbols). | No restrictions on character types; even supports **binary data**. |
| **Primary Use** | Exclusively used to **request** or retrieve data from servers. | Used to **send** complex data to servers and can be used to trigger server-side modifications. |


In the real world, a researcher doesn't always look for `aspirin`. Sometimes, they sketch a molecule in a software, save it as an `.SDF` file, and need to find out: **Does this exist in PubChem, and what are its properties?** 

This notebook teaches you how to answer that question.




## Mastering requests.post()

In this final stage, we transition from individual steps to a production-ready workflow. Sending structural data (`SDF` or `SMILES`) requires a robust approach to handle potential network instabilities or malformed data.

We will consolidate the logic of request handling, error management, and data transformation into a single, seamless workflow. The objective is to implement a **"wrapper" architecture**—a sophisticated coding pattern that acts as a security layer (The Guard) between your local Python environment and the PubChem servers.

**Within this architecture, you will explore:**

* **The Guard** (safe_post): How to define a monitoring function that manages connection health and handles HTTP errors without crashing your session.
* **The Request**: Configuring a secure data payload to retrieve specific properties such as the IUPAC Name and Molecular Weight.
* **The Data Flow**: Streamlining the process where validated raw responses flow directly into a Pandas DataFrame, completing the path from raw structure to structured dataset.

---

### Step 0: Setting the Gateway

Initialize the working environment and define the "address" of the PubChem server. This establishes the foundation for all subsequent programmatic calls to the **PUG-REST** service.

In the code cell below, ensure that the `BASE_URL` is correctly pointed to the PubChem PUG-REST entry point. This variable will be the prefix for every request you build in this notebook.

No visible output is expected yet, but the variables will be stored in your computer's memory. If you run the cell and no error appears, you are ready to start communicating with PubChem.

In [ ]:
# Step 0 
import requests
import pandas as pd
import time
from io import StringIO

# Define the PUG-REST base URL
BASE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

print("Setup complete.")

### Step 1: GET Communication

Before writing code, we must distinguish between the two main "languages" we use to talk to PubChem. While they both retrieve data, they handle the information load differently.

**Concept**:

- **HTTP GET** (The Postcard): You put all your instructions directly into the URL (the address bar). It is simple and fast, but it has a character limit (around 2,048 characters). It is perfect for queries like "Give me the CID for `aspirin`".

- **HTTP POST** (The Parcel): You send the data inside the Request Body (Payload). The URL stays short and clean, while the "heavy" data is hidden inside the message. It is essential for sending complex chemical structures (SMILES/SDF) or massive lists of thousands of compounds.

**Identity vs. Similarity (A Sneak Peek)**:
- Identity Search: "Find me the exact same molecule."
- Similarity Search: "Find me molecules that look like or have a similar skeleton to this one." (We will explore this in later modules).


> **Note**: If you ever see a **414 Request-URI Too Large** error,  
> it means your "postcard" (GET) request was too small for your data.  
> That is the signal to switch to a "parcel" (POST request) instead.

In [ ]:
# Step 1
query_name = "aspirin"

# Build the GET URL to retrieve CIDs from a name
query_url = f"{BASE_URL}/compound/name/{query_name}/cids/JSON"

r = requests.get(query_url)
r.raise_for_status()

data = r.json()

cid = data["IdentifierList"]["CID"][0]
print("CID for aspirin:", cid)

### Step 2: Structure-Based Query Using POST

When we send a chemical structure to PubChem, we are usually performing an **Identity Search**. This is like a fingerprint scan: PubChem looks for a molecule that matches your input exactly in terms of connectivity and atoms.

#### Types of Structural Input:
1.  SMILES (Line Notation): A way of writing a 2D molecule as a single string of text.

* Example 1: `CC(=O)OC1=CC=CC=C1C(=O)O` (aspirin).   
* Example 2: `C[C@H]1[C@H]2CN3C=CC4=C5C=CC=CC5=NC4=C3C[C@@H]2C(=CO1)C(=O)OC` (serpentine).

2. SDF (Structure Data File): A multi-line file format (i.e. the map) that includes either 2D or 3D coordinates of every atom in molecule. It looks like a complex table of numbers and is too large to fit in a URL.

Since SMILES strings can become very long and complex (especially for natural products or polymers), we stop using the URL and start using the **Request Body**.

The **Payload** is simply a dictionary where the keys are the parameters the API expects (like smiles) and the values are your data.

In the upcoming code cell, you will define a chemical structure as a string and configure a `POST` request to ask PubChem: *"Does this structure exist in your database?* *If so, give me its CID.*"

In [ ]:
# Step 2
smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"  # aspirin

# Choose an endpoint for structure identity search
smiles_url = f"{BASE_URL}/compound/smiles/cids/JSON"

# Build payload for POST
payload = {"smiles": smiles}

r = requests.post(smiles_url, data=payload)
r.raise_for_status()

data = r.json()

# Usually, an identity search returns one or a few CIDs
if "IdentifierList" in data:
    cids = data["IdentifierList"]["CID"]
    print(f"Search successful! Found {len(cids)} matching CID(s).")
    print("First 5 CIDs:", cids[:5])
else:
    print("No matches found for this structure.")

print("Returned CIDs:", data["IdentifierList"]["CID"][:5])

### Step 3: Multi-Input Queries

Imagine you have a list of 500 Compound IDs (CIDs) and you need to get their molecular weights.

- **The inefficient way**: Make 500 GET requests (one by one). This is slow and might get your IP temporarily blocked.

- **The professional way**: Make one single POST request sending all 500 IDs in the body.

In PubChem PUG-REST, when we send multiple items, we typically separate them by commas in the payload.

In [ ]:
# Step 3
# Define our target compounds (CIDs)
cid_list = [2244,
            2519, 
            89594, 
            5467006]

# Convert the list to a comma-separated string
cid_string = ",".join(map(str, cid_list))

# Endpoint: notice we use 'cid' in the input domain
cids_url = f"{BASE_URL}/compound/cid/property/MolecularWeight/JSON"

# Build payload for POST
payload = {"cid": cid_string}

r = requests.post(cids_url, data=payload)
r.raise_for_status()

data = r.json()

properties = data["PropertyTable"]["Properties"]

print(f"Sending request for {len(cid_list)} compounds...")

print("\nResults found:")
for item in properties:
    print(f"CID: {item['CID']} -> Molecular Weight: {item['MolecularWeight']} g/mol")

### Step 4: Selecting Multiple Properties (Custom Reports)

One of the most powerful features of PUG-REST is the ability to request a "Custom Report". Instead of making multiple calls for different properties, we can ask for a specific list in a single "Parcel" (POST).

**Key Concept: Property List**
In the URL, we can chain properties separated by commas. This tells PubChem: "Hey, for these CIDs, I want exactly these 5 columns of data."

In [ ]:
# Step 4
# Define our target compounds (CIDs)
cid_list = [2244, 
            2519, 
            89594, 
            5467006]

# Convert the list to a comma-separated string            
cid_string = ",".join(map(str, cid_list))

# Define our target properties
props = [
    "SMILES",
    "InChIKey",
    "MolecularWeight",
    "XLogP",
    "TPSA",
]

# Convert list of properties to a comma-separated string for the URL
props_string = ",".join(props)

# Endpoint: build the URL specifying our custom property list
props_url = f"{BASE_URL}/compound/cid/property/{props_string}/JSON"

# Build payload for POST
payload = {"cid": cid_string}

# Execute the POST request
print(f"Requesting {len(props)} properties for {len(cid_list)} compounds...")

r = requests.post(props_url, data=payload)
r.raise_for_status()

# Process the results
data = r.json()

properties = data["PropertyTable"]["Properties"]

print("\nExample result (First compound):")
for key, value in properties[0].items():
    print(f"{key}: {value}")

### Step 5: From Raw Data to a Research Table

While the JSON response we received in **Step 4** contains all the information, it is hard to read and even harder to analyze. In Data Science, the standard way to handle this is by using the **Pandas** library to create a **DataFrame**.

**Why use a DataFrame?**

- **Readability**: It renders as a clean, sortable HTML table in your Notebook. 
- **Analysis**: You can instantly calculate averages (e.g., average Molecular Weight) or filter compounds (e.g., only those with `XLogP < 2`).
- **Export**: You can save your results to `Excel` or `CSV` with a single line of code.  


Pandas automatically recognizes the keys (CID, SMILES, etc.) as **column headers** and the values as the data in the **rows**.

>**Notice**: We take the list of dictionaries from the previous step.

In [ ]:
# Step 5
import pandas as pd

# Convert to a Pandas DataFrame
df = pd.DataFrame(properties)

# You can reorder columns for better readability (Optional)
column_order = ["CID", "MolecularWeight", "XLogP", "TPSA", "SMILES", "InChIKey"]
df = df[column_order]

# Display the table
print("Final Chemical Database:")
display(df) # In Jupyter, 'display()' renders a nice HTML table


### Step 6: Saving your new data

Running code is great, but real-world research requires sharing your findings. A need step in our pipeline is to "persist" the data—saving it into a file that you can open in **Excel**, **Google Sheets**, or send to a colleague.

**The `to_csv()` Method:**

We use the **CSV (Comma Separated Values)** format because it is universal. However, there are two professional details you should know:

1. `index=False`: By default, Pandas adds a column with row numbers (0, 1, 2...). We usually don't want this in our final Excel file, so we turn it off.

2. `encoding='utf-8'`: Chemical strings (like InChIKeys) can sometimes have special characters. Using `utf-8` ensures your data doesn't get corrupted.


In [ ]:
# Step 6
# Define the filename
filename = "my_first_pubchem_list.csv"

# Export the DataFrame
df.to_csv(filename, index=False, encoding='utf-8')

print(f"Success! Your data has been saved as: {filename}")
print("Check the folder where this notebook is located to find your file.")

# Quick look at the first 2 rows of the file we just saved ---
with open(filename, 'r') as f:
    print("\nPreview of the CSV file content:")
    for _ in range(3):
        print(f.readline().strip())

### Step 7a: Uploading an SDF File

To master PUG-REST, you must understand what you are sending. A **SDF (Structure-Data File)** is a text-based format that describes a molecule's "skeleton."

Now we simulate a real research scenario. Instead of SMILES, we upload a molecular file (`.SDF`).

This requires:

- POST request
- Multipart/form-data
- File upload handling  

If you open an SDF in a text editor, you will see the "Bones and Guts" of the molecule:

- Header: The first 3 lines (Name, source, comments).
- Counts Line: Shows the number of atoms and bonds (e.g., `21 21 ... V2000`).
- Atom Block: Lists every atom with its X, Y, Z coordinates and its element symbol (O, C, H).
- Bond Block: Defines which atom is connected to which, and the type of bond (1=Single, 2=Double). 
- Terminator: The `$$$$` symbols indicate the end of a molecule record.

In this scenario, we will simulate receiving a raw SDF string, saving it as a file, and then using it to query PubChem via **POST**.  


In [ ]:
# Step 7a 
# This raw string is what a chemical drawing software would export.
sdf_content = """Query Molecule
  PubChem

 21 21  0     0  0  0  0  0  0999 V2000
    3.7321   -0.0600    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
    6.3301    1.4400    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
    4.5981    1.4400    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
    2.8660   -1.5600    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
    4.5981   -0.5600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    5.4641   -0.0600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    4.5981   -1.5600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    6.3301   -0.5600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    5.4641   -2.0600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    6.3301   -1.5600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    5.4641    0.9400    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.8660   -0.5600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.0000   -0.0600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    4.0611   -1.8700    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    6.8671   -0.2500    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    5.4641   -2.6800    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    6.8671   -1.8700    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    2.3100    0.4769    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    1.4631    0.2500    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    1.6900   -0.5969    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    6.3301    2.0600    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
  1  5  1  0  0  0  0
  1 12  1  0  0  0  0
  2 11  1  0  0  0  0
  2 21  1  0  0  0  0
  3 11  2  0  0  0  0
  4 12  2  0  0  0  0
  5  6  1  0  0  0  0
  5  7  2  0  0  0  0
  6  8  2  0  0  0  0
  6 11  1  0  0  0  0
  7  9  1  0  0  0  0
  7 14  1  0  0  0  0
  8 10  1  0  0  0  0
  8 15  1  0  0  0  0
  9 10  2  0  0  0  0
  9 16  1  0  0  0  0
 10 17  1  0  0  0  0
 12 13  1  0  0  0  0
 13 18  1  0  0  0  0
 13 19  1  0  0  0  0
 13 20  1  0  0  0  0
M  END
$$$$
"""

# Simulate saving this to a real file
temp_file = "query_molecule.sdf"
with open(temp_file, "w", encoding="utf-8") as f:
    f.write(sdf_content)

print(f"File '{temp_file}' created locally.")

# Sending the FILE content to PubChem via POST
post_url = f"{BASE_URL}/compound/sdf/cids/JSON"

# We read the file we just created to send it in the 'parcel'
with open(temp_file, "r") as f:
    file_payload = {"sdf": f.read()}

r = requests.post(post_url, data=file_payload)
r.raise_for_status()

result_cid = r.json()["IdentifierList"]["CID"][0]
print(f"PubChem recognized this structure! CID: {result_cid}")


### Step 7b: From Local Structure to a Full Technical Sheet

Instead of just identifying the molecule, we will ask PubChem to "read" our local SDF file and return a **complete technical sheet** with its chemical properties.

**How the POST request changes:**

To get specific data, we must modify the Endpoint. Instead of asking for cids, we ask for property/ followed by a list of the features we need, separated by commas.


In [ ]:
# Step 7b
import pandas as pd

# Define the properties we want to retrieve
target_properties = "MolecularFormula,MolecularWeight,IUPACName,SMILES,XLogP,TPSA"

# Update the Endpoint to ask for PROPERTIES instead of just CIDs
full_query_url = f"{BASE_URL}/compound/sdf/property/{target_properties}/JSON"

# Read the local SDF file
with open("query_molecule.sdf", "r") as f:
    sdf_data = f.read()

# Send the POST request
print(f"Requesting properties for the molecule in 'query_molecule.sdf'...")
r = requests.post(full_query_url, data={"sdf": sdf_data})
r.raise_for_status()

# Process the result into a Table
data = r.json()
molecule_properties = data["PropertyTable"]["Properties"]

# Convert to DataFrame for a professional look
df_molecule = pd.DataFrame(molecule_properties)

# Display the final result
print("\nTechnical Sheet Retrieved:")
display(df_molecule)

### Step 7c: Handling Local Files

In many research scenarios, you won't have a list of names or CIDs, but a folder full of SDF files (Structure-Data Files) downloaded from a database or generated by your laboratory.

**The Workflow:**

- **Locate**: We look into the folder `01_Data_Acquisition/data_raw`.
- **Read**: Since PubChem POST can't "see" your hard drive, we must read the content of these files and send the text (the chemical structure) in the payload.
- **Process**: We request the same properties as before.
- **Consolidate**: We merge everything into a single master table.


In [ ]:
# Step 7c
import glob
import os

# Path to our raw data
folder_path = "01_Data_Acquisition/data_raw"
sdf_files = glob.glob(os.path.join(folder_path, "*.sdf"))

print(f"Found {len(sdf_files)} SDF files in the directory.")

# Store each result in a list to create a DataFrame later
all_results = []

# Process each file
for file_path in sdf_files:
    # Read the content of the SDF file (the chemical structure text)
    with open(file_path, 'r') as f:
        sdf_content = f.read()
    
    # Endpoint for SDF identity search (note the change to 'sdf')
    string_url = f"{BASE_URL}/compound/sdf/property/{props_string}/JSON"
    
    # Build payload for POST 
    payload = {"sdf": sdf_content}
    
    try:
        r = requests.post(string_url, data=payload)
        r.raise_for_status()
        
        # Extract properties (usually one molecule per SDF)
        data = r.json()
        molecule_data = data["PropertyTable"]["Properties"][0]
        
        # Add the original filename to track where it came from
        molecule_data["Original_File"] = os.path.basename(file_path)
        all_results.append(molecule_data)
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# Create the final master table
df_final = pd.DataFrame(all_results)

# Save the consolidated results
output_name = "my_rawdata_list.csv"
df_final.to_csv(output_name, index=False)

print(f"\nDone! Consolidated data from {len(all_results)} files into '{output_name}'")
display(df_final.head())

>**Notice**: For this step, we use the `glob` library,   
> which is like a "search engine" for your computer's folders.

### Step 8: Advanced PubChem Searches with ListKey

When performing "heavy" operations—like a Similarity Search (finding molecules that look like yours) or a Substructure Search—PubChem cannot always give you an immediate answer. This way, PubChem cannot return results immediately (e.g., large similarity searches). Instead of CIDs, the server returns a temporary **ListKey** (a job ID).

Instead of waiting for minutes with an open connection, the server uses an **Asynchronous Workflow**:

- The Request: You send your structure via POST.
- The Ticket: PubChem gives you a **ListKey** (a temporary Job ID).
- The Polling: You check back every few seconds to see if the job is finished.
- The Retrieval: Once ready, you use the ListKey to download the final list of CIDs.



In [ ]:
# Step 8 — Submit a "heavy" search (expects ListKey)
import time

# Start a Similarity Search (Finding molecules similar to Aspirin)
similarity_url = f"{BASE_URL}/compound/similarity/smiles/JSON"

payload = {
    "smiles": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Threshold": 95   # It means 95% similarity
}

print("Submitting Similarity Search to PubChem...")
r = requests.post(similarity_url, data=payload)
r.raise_for_status()

# Retrieve the ListKey (Our 'Waiting Ticket')
data = r.json()
listkey = data["Waiting"]["ListKey"]
print(f"Job submitted! Your ListKey is: {listkey}")

# Polling: Checking if the results are ready
results_ready = False
while not results_ready:
    print("Checking status... please wait.")
    
    # Check the ListKey status
    check_url = f"{BASE_URL}/compound/listkey/{listkey}/cids/JSON"
    check_response = requests.get(check_url)
    
    # If the server is still working, it might return a 202 or still have 'Waiting'
    check_data = check_response.json()
    
    if "IdentifierList" in check_data:
        results_ready = True
        print("Results are ready!")
        similar_cids = check_data["IdentifierList"]["CID"]
    else:
        # Wait 3 seconds before checking again to be polite to the server
        time.sleep(3)

# Show the similar compounds found
print(f"Found {len(similar_cids)} molecules similar to Aspirin.")
print("Top 5 CIDs:", similar_cids[:5])

### Step 9: Exporting Search Results

We have a list of several compounds that are chemically similar to `aspirin`. However, a list of numbers (CIDs) isn't very helpful for a chemist. We need to:

- Retrieve properties for all those set of molecules.
- Organize them into a table.
- Save them to a CSV file for further study.


**The "ListKey" Advantage**:  
Instead of sending a massive list of *"N"* IDs back to PubChem, we can simply tell the server: *"Hey, remember that job with ID 16514732469...?* *Give me the Molecular Weight and SMILES for every molecule in that result list."*

In [ ]:
# Step 9
# Define the properties we want for our "Aspirin-like" candidates
screening_props = "SMILES,IUPACName,MolecularWeight,XLogP,TPSA"

# Build the URL using the ListKey instead of CIDs
listkey_url = f"{BASE_URL}/compound/listkey/{listkey}/property/{screening_props}/JSON"

print(f"Requesting details for the {len(similar_cids)} similar compounds...")

# Fetch the data
r = requests.get(listkey_url)
r.raise_for_status()

# Convert to DataFrame
raw_data = r.json()
results_list = raw_data["PropertyTable"]["Properties"]
df_screening = pd.DataFrame(results_list)

# Save to CSV
output_file = "my_first_similarity_results.csv"
df_screening.to_csv(output_file, index=False)

print(f"Success! Data for {len(df_screening)} compounds saved to '{output_file}'")

# Show the first few rows of our new chemical library
display(df_screening.head(5))

### Step 10: Troubleshooting & Error Handling (When POST Fails)

In the real world, APIs don't always return a "200 OK". You might send a malformed SMILES, exceed the server's time limit (e.g. 30-second), or request a resource that doesn't exist. This step teaches you how to "listen" to what the PubChem server is trying to tell you when things go wrong.

**Key Concepts for Debugging:**
When a `requests.post()` call fails, we need to inspect three key things:

1. HTTP Status Codes (The "Check Engine" Lights): 

    - `400 Bad Request`: You likely have a typo in your URL or Payload.
    - `404 Not Found`: The structure or CID you provided is not in the database.
    - `414 URI Too Long`: You used GET for too much data. Switch to POST!
    - `504 Gateway Timeout`: Your request took longer than 30 seconds.


2. `r.text` **vs** `r.json()`: If the server returns an error, it often sends a text message explaining why. If you try to use `.json()` on an error message, your code will crash. Always check the status first!


3. **Timeouts**: To prevent your notebook from freezing if the server is slow, we use the `timeout` parameter.


In the code cell below, you will deliberately trigger a request and practice using `r.status_code` and `r.text` to debug the response. You will also implement a timeout to make your code more robust. Instead of a generic Python error, you should see a clear printout of the Status Code and the Server's explanation of the problem.

In [ ]:
# Step 10
# We will deliberately use an INVALID SMILES to see how the server responds
malformed_smiles = "C1=CC=CC=C1-INVALID-STRUCTURE" 

debugg_url = f"{BASE_URL}/compound/smiles/cids/JSON"

print(f"Testing request with invalid data...")

try:
    # We add 'timeout=10' to wait maximum 10 seconds
    r = requests.post(debugg_url, data={"smiles": malformed_smiles}, timeout=10)
    
    # Check if the request was NOT successful
    if r.status_code != 200:
        print(f"API Error Detected!")
        print(f"Status Code: {r.status_code}")
        
        # Try to show the server's explanation
        # PubChem usually sends a JSON with a 'Fault' message
        try:
            error_details = r.json()
            message = error_details.get("Fault", {}).get("Message", "No detail provided")
            print(f"Server Message: {message}")
        except:
            # If it's not JSON, just print the raw text
            print(f"Server Message: {r.text}")
            
    else:
        print("Success! (Wait, this shouldn't have worked with bad data!)")

except requests.exceptions.Timeout:
    print("Error: The request timed out. The server is too busy.")
except requests.exceptions.RequestException as e:
    print(f"Connection Error: {e}")

### Step 11: Pro-Tip — Building Your Own "Safe" Tools

As your Notebook grows, you will find yourself repeating the same try-except blocks and timeout settings for every request. This makes your code cluttered and hard to maintain.

The professional solution is to create a **Wrapper Function**.

**Why use a Wrapper?**

- DRY (Don't Repeat Yourself): You write the error-handling logic once and reuse it everywhere.
- Cleanliness: Your main analysis cells will only have 1 or 2 lines of code instead of 15.
- Centralized Control: If you want to change the timeout from 30 to 60 seconds for the entire project, you only change it in one place.



In [ ]:
# Step 11
def safe_post(url, payload, timeout=60):
    """
    Sends a POST request and handles common API errors automatically.
    Returns the response object if successful, or raises an error with details.
    """
    try:
        # Execute the request with a safety timeout
        r = requests.post(url, data=payload, timeout=timeout)
        # If the status is NOT 200, this will trigger the 'except' block
        r.raise_for_status()
        return r

    except requests.exceptions.HTTPError:
        print(f"HTTP Error: {r.status_code}")
        # Show the first 500 characters of the server's explanation
        print(f"Server says: {r.text[:500]}")
        raise

    except requests.exceptions.Timeout:
        print("Error: The request timed out. The server is too busy.")
        raise

    except requests.exceptions.RequestException as e:
        print(f"Connection Error: {e}")
        raise

print("Safe POST function is defined and ready to use!")


# --- EXAMPLE OF USE ---
# Now, look how clean your code becomes:
# try:
#    response = safe_post(full_query_url, {"sdf": sdf_data})
#    data = response.json()
#    print("Data retrieved successfully!")
# except:
#    print("The request failed, but we handled it gracefully.")

In [ ]:
my_url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/cids/JSON"
my_data = {"smiles": "C1=CC=CC=C1"} # Benceno

try:
    response = safe_post(my_url, my_data)
    print("Success!")
    print(response.json())
except:
    print("The request failed, but the function already detailed the error in the previous messages.")

### Professional Integration

To conclude our practice, let's put everything into a single, professional block. We will define our `safe_post` function and use it immediately to retrieve data.

**What happens inside this code?**

- The Guard (`safe_post`): It is defined to monitor the connection.
- The Request: We send a request for the IUPAC Name and Molecular Weight.
- The Data Flow: If the "Guard" gives the green light, the data flows into a Pandas DataFrame.




In [ ]:
# Professional Block
import requests
import pandas as pd

# DEFINITION: The "Safety Machine"
def safe_post(url, payload, timeout=20):
    try:
        r = requests.post(url, data=payload, timeout=timeout)
        r.raise_for_status()    # This triggers the except block if status is not 200
        return r
    except Exception as e:
        print(f"Connection managed by safe_post: An error occurred.")
        print(f"Details: {e}")
        return None             # Return None if it fails

# EXECUTION: Using the machine in a real scenario
base_url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/property"
api_url = f"{base_url}/IUPACName,MolecularWeight/JSON"
payload = {"cid": "2244"}

print("--- Starting protected request ---")
response = safe_post(api_url, payload)

# RESULT: If the "Guard" returned a response, we show it
if response:
    data = response.json()
    df = pd.DataFrame(data["PropertyTable"]["Properties"])
    print("Success! The function protected the request and returned valid data:")
    display(df)
else:
    print("The request failed, but your code didn't crash thanks to safe_post.")



### Summary and Next Step

You have successfully extended your molecular data acquisition workflow using PubChem PUG-REST POST requests.

**Current State:**  
We now have the ability to:

- Submit structure-based queries using SMILES strings and SDF files
- Retrieve multiple molecular properties in a single request
- Process more flexible or complex molecular inputs
- Organize PubChem responses into structured research tables
- Handle larger or asynchronous search jobs more robustly

**Next Step:**  
In the next notebook, we will expand our data acquisition workflow beyond PubChem and begin working with ChEMBL, a bioactivity-centered database. There, we will learn how to retrieve targets, assays, activities, and molecular records, and assemble structured datasets suitable for downstream analysis.

---


### EXERCISES

Test your skills with these three challenges. Use the code cells below to write your solutions.

---

**Exercise 1: The Caffeine Challenge**

Caffeine is one of the most studied molecules. Its CID is **2519**.

* Create a script that retrieves its IUPAC Name, Molecular Formula, and Complexity.
* Use a simple requests.get or requests.post.

---

**Exercise 2: Finding "Sisters" of Ibuprofen**

Ibuprofen is a famous anti-inflammatory. We want to find molecules that are 90% similar to it.

* Submit a Similarity Search using the SMILES of Ibuprofen: `CC(C)CC1=CC=C(C=C1)C(C)C(=O)O`.
* Retrieve the ListKey.
* Wait until the results are ready and print the number of similar CIDs found.

---

**Exercise 3: Robust Data Export**

Imagine you have a malformed SMILES: `CC(=O)OC1=CC=CC=C1C(=O)O-WOC.

* Use the safe_post function we created in Step 11 to try and retrieve the CID for this string.
* Your code should not crash. It should print the error message provided by the PubChem server.  

---

**Exercise 4: The Mystery Molecule Challenge**

An old SDF file was found on a lab computer labeled `CT.M1.NB2.unknown_compound.sdf`. Your mission is to use PubChem's API to identify this substance and retrieve its data.

* Create a script to save the "Mystery SDF" string (provided in the `data_raw` folder) into a file named `mystery_compound.sdf`.
To this, copy the coordinates into a variable and save them as `mystery_compound.sdf`.

Then, use your API skills to identify the compound.    
* Use your `safe_post` function to send this file to PubChem.  
* Retrieve and print the **CID**, **IUPAC Name**, **Molecular Formula**, and **Molecular Weight**.  


---

### Outlook

Congratulations! You have completed the professional workflow for interacting with the **PubChem PUG-REST API**. You now possess the skills to:

* Perform complex **POST** requests with chemical structures.
* Automate the creation and reading of **SDF** files.
* Manage large-scale data using **ListKeys**.
* Build **robust and safe functions** to handle real-world API errors. 


See you in the next lesson!

---

### License  
The content of this tutorial itself is licensed under the terms and conditions of the [Creative Commons Attribution (CC BY 4.0) license](https://creativecommons.org/licenses/by/4.0/legalcode.en), and the underlying source code used to format and display that content is licensed under the [MIT license](https://github.com/NanoBiostructuresRG/NumpyTutorial/blob/main/LICENSE). See the LICENSE files for full details.

### Attribution
If you use or adapt this material, please provide appropriate credit to the original [authors](https://orcid.org/my-orcid?orcid=0000-0003-2375-131X) and repository:
[https://github.com/NanoBiostructuresRG](https://github.com/NanoBiostructuresRG)